# # Install dependencies

In [1]:
!pip install -q --user transformers datasets accelerate

import warnings
warnings.filterwarnings("ignore")   # quiet mlflow/sklearn deprecation chatter



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
ERROR: Can not perform a '--user' install. User site-packages are not visible in this virtualenv.


# # Load training data

In [2]:
"""Load the training data.

transformers is NLP-oriented — using an inline sentiment dataset for the
demo. `data.split` shuffles rows by default (use `shuffle=False` to keep
order); `random_state` makes the split reproducible.
"""
import pandas as pd
from nubison_model import data

raw = pd.DataFrame({
    "text": [
        "I love this product", "Amazing experience", "Wonderful day",
        "Best ever made", "Excellent quality", "Pretty good overall",
        "Terrible service", "I hate it", "Awful experience",
        "Worst day ever", "Disappointing result", "Not good at all",
    ],
    "target": [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})

datasets = data.split(raw, {"train": 0.75, "val": 0.25}, random_state=42)
for name, sub in datasets.items():
    print(f"{name:6s} rows={len(sub):3d}")


/home/terry/.cache/pypoetry/virtualenvs/nubison-model-FJ-OInfW-py3.12/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: 

Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.


  color_warning(


train  rows=  9

val    rows=  3

# # Train (HuggingFace transformers)

In [3]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_ID = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2)


def _as_hf_dataset(X, y):
    df = X.assign(labels=y.values.ravel())
    ds = Dataset.from_pandas(df, preserve_index=False)
    return ds.map(
        lambda r: tokenizer(r["text"], padding="max_length", truncation=True, max_length=16),
        batched=True,
    )


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 16558.64it/s]

[transformers] 

DistilBertForSequenceClassification LOAD REPORT

 from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
"""`train(datasets, target, *, ...)` parameters:
    datasets      — dict from `data.split` (must include "train";
                    "val" enables `t.X_val / t.y_val` + auto-scoring).
    target        — label column name(s); single string or list of strings.
    model_type    — "classifier" / "regressor" / free-form string tag.
    weights_path  — default "src/weights.pkl".
"""
from transformers import Trainer, TrainingArguments
from nubison_model import train

with train(datasets=datasets, target=["target"], model_type="classifier") as t:
    train_ds = _as_hf_dataset(t.X_train, t.y_train)
    val_ds = _as_hf_dataset(t.X_val, t.y_val)

    args = TrainingArguments(
        output_dir="/tmp/hf_trainer",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        eval_strategy="epoch",
        logging_steps=1,
        report_to=["mlflow"],
        disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds)
    trainer.train()

    t.save(model)

print(f"run_id: {t.run_id}")


2026/05/15 22:39:25 INFO mlflow.tracking.fluent: Experiment with name 'NoWarn_1778852317_train_transformers' does not exist. Creating a new experiment.


2026/05/15 22:39:25 INFO mlflow.bedrock: Enabled auto-tracing for Bedrock. Note that MLflow can only trace boto3 service clients that are created after this call. If you have already created one, please recreate the client by calling `boto3.client`.


2026/05/15 22:39:25 INFO mlflow.tracking.fluent: Autologging successfully enabled for boto3.


2026/05/15 22:39:26 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


2026/05/15 22:39:26 INFO mlflow.tracking.fluent: Autologging successfully enabled for transformers.


...

{'eval_loss': '0.6895', 'eval_runtime': '0.0157', 'eval_samples_per_second': '190.7', 'eval_steps_per_second': '63.55', 'epoch': '2'}

{'train_runtime': '1.597', 'train_samples_per_second': '11.27', 'train_steps_per_second': '3.756', 'train_loss': '0.6819', 'epoch': '2'}

🏃 View run efficient-shoat-117 at: http://127.0.0.1:5050/#/experiments/37/runs/a5b564a8211a4e6f8de8cdc8028bf87e


🧪 View experiment at: http://127.0.0.1:5050/#/experiments/37


run_id: a5b564a8211a4e6f8de8cdc8028bf87e